# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-af3368c2-0c72-4ac9-b347-6633cec5adbb;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 440ms :: artifacts dl 21ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de nos hypothèses:**

• **Test #1 (Mann-Whitney U) :** $H_0 =$ `La distribution des montants de transaction (amount_usd) est identique entre les paiements physiques (puce/piste) et les paiements en ligne (is_physical = True vs False) / $H_1 =$ = Les distributions des montants diffèrent significativement selon le canal de paiement. (Justification du choix du test : La variable amount_usd étant fortement asymétrique avec un SKEWNESS de 8,1, le test non-paramétrique de Mann-Whitney U compare les rangs et médianes sans hypothèse de normalité).`
• **Test #2 (Test du χ² d'indépendance) :** $H_0 =$ `La survenue d'une erreur technique (has_error) est statistiquement indépendante du mode de paiement utilisé (use_chip) / $H_1 =$ = Il existe une dépendance significative entre le mode de paiement et le taux d'erreur. (Justification du choix du test : Les deux variables étudiées étant qualitatives catégorielles, le test du χ² permet d'évaluer l'association entre leurs fréquences observées).`

In [2]:
SILVER_MAIN_PATH = "/lakehouse/silver/transactions"

# Projection légère côté Spark (uniquement les colonnes utiles + échantillon représentatif de 200 000 lignes pour le test de rangs)
df_silver = spark.read.format("delta").load(SILVER_MAIN_PATH)

# 1. Données pour Test #1 (Mann-Whitney U sur échantillon aléatoire de 200k transactions pour préserver la RAM du driver)
df_sample = (
    df_silver.select("amount_usd", "is_physical")
    .sample(withReplacement=False, fraction=0.02, seed=42)
    .toPandas()
)

assert df_sample is not None, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."

groupe_physique = df_sample[df_sample["is_physical"] == True]["amount_usd"]
groupe_online = df_sample[df_sample["is_physical"] == False]["amount_usd"]

stat_mw, p_val_mw = stats.mannwhitneyu(groupe_physique, groupe_online, alternative="two-sided")
print(f"=== Test #1 : Mann-Whitney U (Physique vs En ligne) ===")
print(
    f"Médiane Physique : {groupe_physique.median():.2f} $ | Médiane En ligne : {groupe_online.median():.2f} $"
)
print(f"Statistique U = {stat_mw:,.0f} | p-value = {p_val_mw:.4e}\n")

# 2. Données pour Test #2 (Chi-2 calculé sur l'intégralité des 12,6M de lignes via agrégation Spark préalable)
contingence_pd = (
    df_silver.groupBy("use_chip", "has_error")
    .count()
    .toPandas()
    .pivot(index="use_chip", columns="has_error", values="count")
    .fillna(0)
)

chi2_stat, p_val_chi2, dof, _ = stats.chi2_contingency(contingence_pd)
print(f"=== Test #2 : Chi-2 d'indépendance (Mode de paiement vs Erreur) ===")
print(contingence_pd)
print(f"Statistique Chi2 = {chi2_stat:.2f} (ddl={dof}) | p-value = {p_val_chi2:.4e}")


26/11/20 13:05:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== Test #1 : Mann-Whitney U (Physique vs En ligne) ===
Médiane Physique : 30.90 $ | Médiane En ligne : 35.17 $
Statistique U = 2,983,858,611 | p-value = 5.2212e-296



=== Test #2 : Chi-2 d'indépendance (Mode de paiement vs Erreur) ===
has_error                 0      1
use_chip                          
Chip Transaction    4444033  68234
Online Transaction  1515692  35533
Swipe Transaction   6473722  98014
Statistique Chi2 = 5421.86 (ddl=2) | p-value = 0.0000e+00


**Conclusion des tests d'inférence statistique :**
> `Les deux tests non-paramétriques aboutissent au rejet catégorique des hypothèses nulles H₀ au seuil de significativité α = 5% (p-value < 0,0001). D'une part, le test de Mann-Whitney U confirme une différence statistiquement significative entre la distribution des montants réglés physiquement et celle des paiements en ligne. D'autre part, le test du χ² d'indépendance (χ² = 5 421,86 pour 2 degrés de liberté) démontre une forte dépendance entre le canal de paiement et la survenue d'anomalies techniques : les transactions en ligne (Online Transaction) présentent un taux d'erreur de 2,29 %, nettement supérieur à celui observé sur les paiements physiques par puce ou piste magnétique (1,51 %).`

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [5]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/segments_clients"

# 0. Enregistrement de la table Silver en vue SQL dans cette session
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")

# 1. Agrégation distribuée par client côté PySpark (4 features comportementales)
entities_spark = spark.sql("""
    SELECT
        client_id,
        COUNT(*) AS frequence_tx,
        ROUND(SUM(amount_usd), 2) AS volume_total_usd,
        ROUND(PERCENTILE_APPROX(amount_usd, 0.50), 2) AS panier_median_usd,
        ROUND(AVG(CAST(is_physical AS INT)) * 100, 2) AS pct_paiement_physique
    FROM v_silver
    GROUP BY client_id
    HAVING COUNT(*) >= 5
""")

entities_pd = entities_spark.toPandas()
assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."

# 2. Prétraitement statistique : log1p sur les variables monétaires/fréquences asymétriques + StandardScaler
features_cols = ["frequence_tx", "volume_total_usd", "panier_median_usd", "pct_paiement_physique"]
X_transformed = entities_pd[features_cols].copy()
for col in ["frequence_tx", "volume_total_usd", "panier_median_usd"]:
    X_transformed[col] = np.log1p(X_transformed[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformed)

# Entraînement K-Means à 4 segments
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Inspection des médianes par cluster pour labellisation automatique cohérente
profils = entities_pd.groupby("cluster_id")[features_cols].median()
print("=== Profils médians par Cluster K-Means ===")
print(profils, "\n")

# Attribution intelligente des labels métier selon le volume total
order_by_vol = profils["volume_total_usd"].sort_values().index.tolist()
label_map = {
    order_by_vol[0]: "Petits Porteurs Occasionnels",
    order_by_vol[1]: "Clients Réguliers Standards",
    order_by_vol[2]: "Clients Actifs à Fort Volume",
    order_by_vol[3]: "Clients VIP / Gros Dépensiers",
}
entities_pd["segment"] = entities_pd["cluster_id"].map(label_map)

assert "segment" in entities_pd.columns, (
    "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."
)

# 3. Sauvegarde en Delta Lake sur HDFS dans GOLD_SEGMENTS_PATH
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} clients) !")


=== Profils médians par Cluster K-Means ===
            frequence_tx  volume_total_usd  panier_median_usd  \
cluster_id                                                      
0                11773.0        683375.840              43.21   
1                 9273.5        339898.075              14.02   
2                14665.0        666002.940              30.98   
3                 5444.0        312282.010              36.44   

            pct_paiement_physique  
cluster_id                         
0                           94.44  
1                           94.68  
2                           62.44  
3                           91.55   



Table de segmentation enregistrée dans /lakehouse/gold/segments_clients (1,219 clients) !


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [6]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/evolution_mensuelle       :       118 lignes


 - /lakehouse/gold/pareto_mcc                :       109 lignes


 - /lakehouse/gold/segments_clients          :     1,219 lignes
 - /lakehouse/gold/top_mcc_par_etat          :       200 lignes
